# Semantic Similarity Analysis - AI Actors Network

Ce notebook mesure la proximite semantique entre entreprises a partir des descriptions textuelles.

## Objectif

Evaluer la proximite semantique des acteurs IA et produire une carte 2D exploitable pour lecture metier.

## Pipeline

1. Extraire les entreprises avec description non vide et valorisation > 100 M USD.
2. Generer des embeddings multilingues.
3. Calculer la similarite cosine puis la distance.
4. Projeter en 2D avec MDS.
5. Visualiser, analyser les paires proches et exporter les artefacts.

## Convention de lecture

Chaque section suit le schema: **Objectif -> Entrees -> Traitement -> Sorties**.

## Regles clefs

- `valuation = max(capitalization, funds_raised)` en millions.
- Distance semantique: `1 - cosine_similarity`.
- MDS 2D avec seed fixe pour reproductibilite.

## 1. Configuration et imports

**Objectif**: charger les dependances NLP/ML et definir les chemins de travail.

**Entrees**: environnement Python + `database.db`.
**Sorties**: constantes (`DB_PATH`, `EXPORTS_DIR`, `RANDOM_SEED`) et modules importes.

In [7]:
import sqlite3
from pathlib import Path
import warnings
import sys
import subprocess
import importlib

import numpy as np
import pandas as pd
import plotly.express as px

# Auto-install missing ML dependencies when needed
REQUIRED_PKGS = {
    "sentence_transformers": "sentence-transformers",
    "sklearn": "scikit-learn",
}

for module_name, pip_name in REQUIRED_PKGS.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installation de {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

from sentence_transformers import SentenceTransformer
from sklearn.manifold import MDS
from sklearn.metrics.pairwise import cosine_similarity

ROOT        = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DB_PATH     = ROOT / "database.db"
EXPORTS_DIR = ROOT / "analyses" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

print(f"Base : {DB_PATH}")
print(f"Exports : {EXPORTS_DIR}")

Base : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\database.db
Exports : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports


## 2. Extraction SQL des entreprises

**Objectif**: constituer le jeu d'entreprises analyse.

**Criteres**:
- description non vide;
- valorisation strictement superieure a 100 M USD.

**Sorties**: `df_raw` + export `semantic_raw.csv`.

In [8]:
SQL = """
SELECT
    id,
    name,
    description,
    sector,
    country,
    founded_year,
    MAX(
        CAST(REPLACE(IFNULL(capitalization, '0'), ',', '.') AS REAL),
        CAST(REPLACE(IFNULL(funds_raised,    '0'), ',', '.') AS REAL)
    ) AS valuation
FROM enterprises
WHERE description IS NOT NULL
  AND description != ''
  AND MAX(
        CAST(REPLACE(IFNULL(capitalization, '0'), ',', '.') AS REAL),
        CAST(REPLACE(IFNULL(funds_raised,    '0'), ',', '.') AS REAL)
      ) > 100
ORDER BY valuation DESC
"""

with sqlite3.connect(DB_PATH) as con:
    df_raw = pd.read_sql_query(SQL, con)

print(f"{len(df_raw)} entreprises sélectionnées")
print(f"Valorisation min : {df_raw['valuation'].min():.0f}M | max : {df_raw['valuation'].max():.0f}M")
df_raw[["name", "valuation", "description"]].head(10)

84 entreprises sélectionnées
Valorisation min : 103M | max : 400000M


,name,valuation,description
0,Berkshire Hathaway,400000.0,"Berkshire Hathaway, originally founded in 1955..."
1,ByteDance,50000.0,ByteDance is a global internet company founded...
2,OpenAI,30000.0,OpenAI est un laboratoire de recherche et une ...
3,Alibaba,30000.0,Alibaba Group is a Chinese technology company ...
4,Waymo,27100.0,Filiale of Alphabet. Value proposition: a turn...
5,Databricks,20000.0,Fondée en 2013 par les créateurs d'Apache Spar...
6,Scale AI,16000.0,est l'un des piliers invisibles mais indispens...
7,Anthropic,10000.0,Societe d'IA generative (famille de modeles Cl...
8,Cerebras,8500.0,"Cerebras Systems, a pioneering manufacturer of..."
9,Nscale,3700.0,1. La location de supercalculateurs et de GPUN...


In [9]:
# Export des données brutes
raw_path = EXPORTS_DIR / "semantic_raw.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Export brut → {raw_path}")

Export brut → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\semantic_raw.csv


## 3. Embeddings multilingues

**Objectif**: encoder chaque description en vecteur dense comparable.

**Modele**: `paraphrase-multilingual-MiniLM-L12-v2` (multilingue).
**Entree**: descriptions de `df_raw`.
**Sortie**: matrice `embeddings` de dimension `(N, 384)`.

In [10]:
# Chargement du modèle multilingue
print("Chargement du modèle sentence-transformers...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"Modèle chargé : {model.get_sentence_embedding_dimension()} dimensions")

# Génération des embeddings
descriptions = df_raw['description'].tolist()
print(f"Génération des embeddings pour {len(descriptions)} descriptions...")
embeddings = model.encode(descriptions, show_progress_bar=True, batch_size=32)

print(f"Embeddings shape : {embeddings.shape}")
print(f"Exemple (première entreprise) : {embeddings[0][:5]}... (5 premières dimensions)")

Chargement du modèle sentence-transformers...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modèle chargé : 384 dimensions
Génération des embeddings pour 84 descriptions...


C:\Users\33623\AppData\Local\Temp\ipykernel_39692\1891095489.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Modèle chargé : {model.get_sentence_embedding_dimension()} dimensions")


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings shape : (84, 384)
Exemple (première entreprise) : [-0.02192629 -0.05687128 -0.16240461 -0.10960551 -0.0142429 ]... (5 premières dimensions)


## 4. Similarite cosine et matrice de distance

**Objectif**: convertir la proximite semantique en distance exploitable pour MDS.

**Traitement**:
- calcul de la matrice de similarite cosine;
- transformation en distance `1 - similarite`;
- diagonale forcee a 0.

**Sortie**: `distance_matrix` + export `semantic_distance_matrix.csv`.

In [11]:
# Calcul de la similarité cosine
similarity_matrix = cosine_similarity(embeddings)
print(f"Matrice de similarité : {similarity_matrix.shape}")
print(f"Similarité min : {similarity_matrix.min():.3f} | max : {similarity_matrix.max():.3f}")

# Conversion en distance
distance_matrix = 1 - similarity_matrix
np.fill_diagonal(distance_matrix, 0)

print(f"Matrice de distance : {distance_matrix.shape}")
print(f"Distance min (hors diagonale) : {distance_matrix[distance_matrix > 0].min():.3f}")
print(f"Distance max : {distance_matrix.max():.3f}")

# Export de la matrice de distance
df_distance = pd.DataFrame(
    distance_matrix,
    index=df_raw['name'],
    columns=df_raw['name']
)
distance_path = EXPORTS_DIR / "semantic_distance_matrix.csv"
df_distance.to_csv(distance_path)
print(f"Export matrice de distance → {distance_path}")

Matrice de similarité : (84, 84)
Similarité min : -0.160 | max : 1.000
Matrice de distance : (84, 84)
Distance min (hors diagonale) : 0.005
Distance max : 1.160
Export matrice de distance → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\semantic_distance_matrix.csv


## 5. Projection MDS 2D

**Objectif**: representer les proximites semantiques dans un plan 2D.

**Entree**: `distance_matrix` pre-calcullee.
**Parametres**: `n_components=2`, `dissimilarity='precomputed'`, seed fixe.
**Sortie**: `df_coords` + export `semantic_coords_2d.csv`.

In [12]:
N = len(df_raw)
print(f"Projection MDS pour {N} entreprises...")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        init="random",
        random_state=RANDOM_SEED,
        normalized_stress="auto"
    )
    coords = mds.fit_transform(distance_matrix)

print(f"MDS stress : {mds.stress_:.2f}")
print(f"Coordonnées shape : {coords.shape}")

# Construction du DataFrame de coordonnées
df_coords = pd.DataFrame({
    "name": df_raw['name'],
    "x": coords[:, 0],
    "y": coords[:, 1],
    "valuation": df_raw['valuation'],
    "log_valuation": np.log10(df_raw['valuation'] + 1),
    "sector": df_raw['sector'].fillna("Unknown"),
    "country": df_raw['country'].fillna("Unknown"),
    "description": df_raw['description']
})

# Extraction du secteur principal
df_coords["sector"] = df_coords["sector"].apply(
    lambda s: s.split(",")[0].strip() if pd.notna(s) and s != "Unknown" else "Unknown"
)

coords_path = EXPORTS_DIR / "semantic_coords_2d.csv"
df_coords.to_csv(coords_path, index=False)
print(f"Coordonnées 2D → {coords_path}")

df_coords.sort_values("valuation", ascending=False).head(10)

Projection MDS pour 84 entreprises...
MDS stress : 203.45
Coordonnées shape : (84, 2)
Coordonnées 2D → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\semantic_coords_2d.csv


,name,x,y,valuation,log_valuation,sector,country,description
0,Berkshire Hathaway,-0.689299,-0.282172,400000.0,5.602061,Financial Services,Unknown,"Berkshire Hathaway, originally founded in 1955..."
1,ByteDance,0.216266,-0.506169,50000.0,4.698979,Media & Entertainment,China,ByteDance is a global internet company founded...
2,OpenAI,0.025793,-0.096219,30000.0,4.477136,Unknown,Unknown,OpenAI est un laboratoire de recherche et une ...
3,Alibaba,-0.443318,0.343348,30000.0,4.477136,Cloud Provider,China,Alibaba Group is a Chinese technology company ...
4,Waymo,0.274000,0.522785,27100.0,4.432985,Transport & Mobility,USA,Filiale of Alphabet. Value proposition: a turn...
5,Databricks,-0.287827,0.254458,20000.0,4.301052,ICT,USA,Fondée en 2013 par les créateurs d'Apache Spar...
6,Scale AI,-0.065416,0.064741,16000.0,4.204147,ICT,USA,est l'un des piliers invisibles mais indispens...
7,Anthropic,-0.152271,-0.072613,10000.0,4.000043,Hardware,USA,Societe d'IA generative (famille de modeles Cl...
8,Cerebras,-0.150113,-0.311875,8500.0,3.929470,Hardware,USA,"Cerebras Systems, a pioneering manufacturer of..."
9,Nscale,-0.074093,-0.377273,3700.0,3.568319,Cloud Provider,UK,1. La location de supercalculateurs et de GPUN...


## 6. Visualisation interactive

**Objectif**: produire la carte 2D lisible pour exploration metier.

**Encodages visuels**:
- couleur: secteur principal;
- taille: log de la valorisation;
- hover: nom, secteur, pays, valorisation, extrait description.

**Sortie**: export HTML `semantic_similarity_map_2d.html`.

In [13]:
# Préparation du hover text
def fmt_hover(row):
    lines = [f"<b>{row['name']}</b>"]
    if pd.notna(row.get("sector")) and row["sector"] != "Unknown":
        lines.append(f"Sector: {row['sector']}")
    if pd.notna(row.get("country")) and row["country"] != "Unknown":
        lines.append(f"Country: {row['country']}")
    val = float(row.get("valuation") or 0)
    if val > 1000:
        lines.append(f"Valuation: {val/1000:.1f}B USD")
    else:
        lines.append(f"Valuation: {val:.0f}M USD")
    if pd.notna(row.get("description")):
        desc = str(row["description"])
        snippet = desc[:200].rstrip()
        lines.append(f"<i>{snippet}{'…' if len(desc) > 200 else ''}</i>")
    return "<br>".join(lines)

df_coords["hover"] = df_coords.apply(fmt_hover, axis=1)
df_coords["marker_size"] = np.maximum(df_coords["log_valuation"] * 3, 5)

fig = px.scatter(
    df_coords,
    x="x", y="y",
    color="sector",
    size="marker_size",
    size_max=25,
    text="name",
    custom_data=["hover"],
    color_discrete_sequence=px.colors.qualitative.Light24,
    title=f"Similarité sémantique — MDS ({N} entreprises · taille ∝ log valorisation)",
    labels={"x": "Dimension 1", "y": "Dimension 2", "sector": "Secteur"},
    width=1400,
    height=1200,
)

fig.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>",
    textposition="top center",
    textfont=dict(size=7),
    marker=dict(opacity=0.78, line=dict(width=0.4, color="white")),
)

fig.update_layout(
    legend=dict(title="Secteur", font=dict(size=10)),
    font=dict(family="Inter, sans-serif", size=11),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
)

fig_html = EXPORTS_DIR / "semantic_similarity_map_2d.html"
fig.write_html(str(fig_html))
print(f"Carte interactive → {fig_html}")
fig.show()

Carte interactive → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\semantic_similarity_map_2d.html


## 7. Paires les plus proches

**Objectif**: identifier les binomes d'entreprises les plus similaires semantiquement.

**Sortie**: `df_pairs` trie par similarite decroissante + export `semantic_similarity_pairs.csv`.

In [14]:
# Top 20 paires les plus similaires
n = len(df_raw)
pairs = []

for i in range(n):
    for j in range(i+1, n):
        pairs.append({
            "company_1": df_raw.iloc[i]['name'],
            "company_2": df_raw.iloc[j]['name'],
            "similarity": similarity_matrix[i, j],
            "distance": distance_matrix[i, j]
        })

df_pairs = pd.DataFrame(pairs).sort_values("similarity", ascending=False)

print("Top 20 paires les plus similaires :")
df_pairs.head(20)

Top 20 paires les plus similaires :


,company_1,company_2,similarity,distance
3441,Xelix,SiPearl,0.994688,0.005312
3447,Xelix,Nabla,0.993914,0.006086
3454,SiPearl,Tessl,0.992768,0.007232
3455,SiPearl,Nabla,0.989634,0.010366
3446,Xelix,Tessl,0.989185,0.010815
3480,Tessl,Nabla,0.987822,0.012178
529,Scale AI,Hugging Face,0.839527,0.160473
1612,Perplexity,Perplexity AI,0.836913,0.163087
1302,MiniMax,Hailo Technologies,0.816326,0.183674
3348,BuildOps,PlanRadar,0.753639,0.246361


In [15]:
# Export des paires
pairs_path = EXPORTS_DIR / "semantic_similarity_pairs.csv"
df_pairs.to_csv(pairs_path, index=False)
print(f"Export paires de similarité → {pairs_path}")

Export paires de similarité → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\semantic_similarity_pairs.csv


## 8. Recapitulatif des exports

Cette section liste les artefacts produits et leur taille pour verification rapide de fin de pipeline.

In [16]:
print("── Récapitulatif des exports ────────────────────────────")
for p in [raw_path, distance_path, coords_path, pairs_path, fig_html]:
    size_kb = Path(p).stat().st_size / 1024
    print(f"  {p.name:<42} {size_kb:6.1f} KB")

── Récapitulatif des exports ────────────────────────────
  semantic_raw.csv                             78.6 KB
  semantic_distance_matrix.csv                 72.7 KB
  semantic_coords_2d.csv                       81.7 KB
  semantic_similarity_pairs.csv               144.3 KB
  semantic_similarity_map_2d.html            4789.3 KB
